In [ ]:
%load_ext autoreload
%autoreload 2
import os

os.chdir("..")
print(os.getcwd())

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import torch
import wandb

In [ ]:
def pprint(dictionary):
    for k, v in dictionary.items():
        print(k, ":", v)

In [ ]:
os.makedirs("outputs/alignment", exist_ok=True)

# Table prep functions

In [ ]:
def format_mean_sem(mean, sem, decimals=3, scale=1.0, math_mode=True):
    """Format a single mean/SEM pair as a LaTeX string, with optional scaling."""
    if pd.isna(mean):
        return "--"
    m = mean * scale
    if pd.isna(sem):
        s = f"{m:.{decimals}f}"
    else:
        s = f"{m:.{decimals}f} $\\pm$ {sem * scale:.{decimals}f}"
    return f"{s}" if math_mode else s


def add_mean_sem_columns(df, metrics, math_mode=True):
    """
    metrics: dict mapping output_col_name -> dict with:
        mean_col, sem_col, decimals (int), scale (float, default 1.0)
    """
    df = df.copy()
    for out_col, spec in metrics.items():
        df[out_col] = [
            format_mean_sem(
                m,
                s,
                decimals=spec.get("decimals", 3),
                scale=spec.get("scale", 1.0),
                math_mode=math_mode,
            )
            for m, s in zip(df[spec["mean_col"]], df[spec["sem_col"]])
        ]
    return df


def get_latex(df, columns=None, escape=False):
    if columns is not None:
        df = df[columns]
    else:
        columns = list(df.columns)

    latex_table = df.to_latex(
        index=False,
        columns=columns,
        escape=escape,  # False: let \pm and $ pass through untouched
    )
    print(latex_table)

    payload = {
        "formula": latex_table,
        "fsize": "54px",
        "fcolor": "000000",
        "mode": "0",
        "out": "1",
        "remhost": "quicklatex.com",
        "preamble": r"\usepackage{booktabs}\usepackage{amsmath}",
    }
    response = requests.post("https://quicklatex.com/latex3.f", data=payload)
    print(response.text)
    return latex_table


def parse_modality(x, modalities):
    for k, v in modalities.items():
        if k in x:
            return v

# WANDB API parsing

In [ ]:
# Connect to wandb api
api = wandb.Api()
entity = "aether_xai"

In [ ]:
project = "s2bms_alignment"  ## textual / both / alignment_lv5
results_df_path = "outputs/alignment/results_s2bms.csv"

# S2BMS

## csv log

In [ ]:
# if os.path.exists(results_df_path):
#     df = pd.read_csv(results_df_path)
#     print(f"There are {len(df)} records already in results.")
#     df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
# else:
#     df = None

df = None

# Separate concept columns

In [ ]:
import json

# Collect columns into 3 categories per version
versions = {"v11": {}, "v10": {}}

for k, v in versions.items():
    with open(f"aether/data/s2bms/concept_captions/{k}.json", "r") as f:
        data = json.load(f)

    v["in_dist_valid"] = []
    v["in_dist_unvalid"] = []
    v["out_dist"] = []

    for d in data:
        if "test" in d["col"]:
            name = d["col"].replace("test-", "test_dyn_k_index_")
            name = f"{name.replace('aux_', '')}_{'max' if d['is_max'] else 'min'}"
            v["out_dist"].append(name)
        else:
            name = f"{d['col'].replace('aux_', 'test_dyn_k_index_')}_{'max' if d['is_max'] else 'min'}"
            if d["val_av"]:
                v["in_dist_valid"].append(name)
            else:
                v["in_dist_unvalid"].append(name)

## fetching

In [ ]:
def log_concept_avr(run, cols, name):
    metrics = dict(run.summary)
    matched_metrics = [metrics[c] for c in cols]
    value = sum(matched_metrics) / len(matched_metrics)
    run.summary.update({name: value})
    print(name, value)


def flatten(d: dict, parent_key: str = "", sep: str = ".") -> dict:
    """Flatten a nested config dict (e.g. a Hydra config) into dot-separated keys."""
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten(v, new_key, sep=sep))
        else:
            items[new_key] = v
    return items

In [ ]:
# project = "s2bms_vlm_baselines"

In [ ]:
api = wandb.Api()
runs_iterator = api.runs(f"{entity}/{project}")
run_list = []

for i, run in enumerate(runs_iterator):
    if run.state != "finished":
        print(f"{run.id} is not finished.")
        continue
    concept_v = run.config["data"]["caption_builder"]["concepts_fname"].replace(".json", "")
    if concept_v == "${version}":
        concept_v = run.summary["experiment"]
        match = re.search(r"(?<=_)v\d+", concept_v)
        concept_v = match.group() if match else None

    if run.summary.get("test_avr_top-dyn_k_index_in_dist_valid") is None:
        try:
            in_dist_valid = versions[concept_v]["in_dist_valid"]
            in_dist_unvalid = versions[concept_v]["in_dist_unvalid"]
            out_dist = versions[concept_v]["out_dist"]

            log_concept_avr(run, in_dist_valid, "test_avr_top-dyn_k_index_in_dist_valid")
            log_concept_avr(run, in_dist_unvalid, "test_avr_top-dyn_k_index_in_dist_unvalid")
            log_concept_avr(run, out_dist, "test_avr_top-dyn_k_index_ood")
        except Exception as e:
            if concept_v == "v8":
                print(f"v8 issue {run.id}: {e}")
            else:
                print(f"{run.id}: {e}")

    if df is not None:
        if run.id in list(df.run_id):
            print(
                f'{run.summary["experiment"]} with seed={run.config["seed"]} already in results.'
            )
            continue

    temp = dict(run.summary)
    config_dict = flatten(run.config)
    captures = {}
    captures["experiment"] = run.summary["experiment"]
    captures["run_id"] = run.id
    for k, v in temp.items():
        # if "test" not in k and "best" not in k:
        #     captures[f"best_{k}"] = v
        # else:
        #     captures[k] = v
        if "test" in k and "dyn_k_index" in k:
            captures[k] = v

    captures = {**captures, **config_dict}
    del captures["tags"]

    run_list.append(captures)
    print(f'{run.summary["experiment"]} with seed={run.config["seed"]} logged.')

In [ ]:
list_aef = []
list_other = []

for r in run_list:
    if r["experiment"].startswith("aef"):
        list_aef.append(r)
    else:
        list_other.append(r)
print(f"len(list_aef)={len(list_aef)}, len(list_other)={len(list_other)}")
df_dict = {}
df_dict["aef"] = pd.DataFrame(list_aef)
if len(list_other) > 0:
    df_dict["baseline"] = pd.DataFrame(list_other)

import re

for k, v in df_dict.items():
    test_cols = [c for c in v.columns if "test" in c and "dyn_k_index" in c]

    dict_cols = {}

    dict_cols["bioclim"] = [c for c in test_cols if "bioclim" in c]
    dict_cols["corine_all"] = [c for c in test_cols if "corine" in c]
    cols_corine_all = dict_cols["corine_all"]
    pattern = re.compile(r"corine_frac_\d_")
    dict_cols["corine_high"] = [s for s in cols_corine_all if pattern.search(s)]
    pattern = re.compile(r"corine_frac_\d\d_")
    dict_cols["corine_mid"] = [s for s in cols_corine_all if pattern.search(s)]
    pattern = re.compile(r"corine_frac_\d\d\d_")
    dict_cols["corine_low"] = [s for s in cols_corine_all if pattern.search(s)]
    dict_cols["corine_urban"] = [s for s in cols_corine_all if "corine_frac_1" in s]
    dict_cols["corine_agri"] = [s for s in cols_corine_all if "corine_frac_2" in s]
    dict_cols["corine_forest"] = [s for s in cols_corine_all if "corine_frac_3" in s]
    dict_cols["corine_wetland"] = [s for s in cols_corine_all if "corine_frac_4" in s]
    dict_cols["corine_water"] = [s for s in cols_corine_all if "corine_frac_5" in s]
    dict_cols["hf"] = [
        "test_dyn_k_index_maxdist_road_max",
        "test_dyn_k_index_pop_density_max",
        "test_dyn_k_index_total_population_max",
        "test_dyn_k_index_meandist_road_max",
    ]
    dict_cols["ukceh"] = [s for s in test_cols if "index_p_" in s]
    dict_cols["pollution"] = ["test_dyn_k_index_pm252024g_max"]
    dict_cols["topo"] = [
        "test_dyn_k_index_river_dens_max",
        "test_dyn_k_index_dist_to_sea_max",
        "test_dyn_k_index_elevation_max",
        "test_dyn_k_index_slope_max",
    ]
    dict_cols["lc_het"] = ["test_dyn_k_index_shdi_lc_max"]

    cols_groups = []
    for k2, v2 in dict_cols.items():
        new_col = f"test_avr_top-dyn_k_index_aggr_{k2}"
        df_dict[k][new_col] = df_dict[k][v2].mean(axis=1)
        cols_groups.append(new_col)
    df_dict[k]["group_av"] = df_dict[k][cols_groups].mean(axis=1)

    cols_drop = []

    for c in v.columns:
        if any([isinstance(x, list) for x in v[c]]):
            cols_drop.append(c)
        elif v[c].nunique() <= 1:
            cols_drop.append(c)

    df_dict[k].drop(columns=cols_drop, inplace=True)

## AEF

In [ ]:
## Selection
metrics_use = [
    "test_avr_top-dyn_k_index_in_dist_valid",
    "test_avr_top-dyn_k_index_in_dist_unvalid",
    "test_avr_top-dyn_k_index_ood",
]
metrics_use += cols_groups + ["group_av"]
n_cols = 4

n_metrics = len(metrics_use)  # +1 for average
n_rows = n_metrics // n_cols + (n_metrics % n_cols > 0)

for use_unlabelled_data in [True, False]:
    for version in ["v10", "v11"]:
        runs_sel = df_dict["aef"].copy()
        runs_sel = runs_sel[runs_sel["data.caption_builder.concepts_fname"] == f"{version}.json"]

        # for c in [
        #     "data.dataset.use_unlabelled_data",
        #     "data.batch_size",
        #     "seed",
        #     "model.loss_fn.sigma",
        #     "optimizer.lr",
        # ]:
        #     print(c, runs_sel[c].unique())

        if "optimizer.lr" in runs_sel.columns:
            runs_sel = runs_sel[runs_sel["optimizer.lr"] == 0.001]
        if "use_unlabelled_data" in runs_sel.columns:
            runs_sel = runs_sel[
                runs_sel["data.dataset.use_unlabelled_data"] == use_unlabelled_data
            ]

        cols_drop = []
        for c in runs_sel.columns:
            if any([isinstance(x, list) for x in runs_sel[c]]):
                cols_drop.append(c)
            elif runs_sel[c].nunique() <= 1:
                cols_drop.append(c)

        runs_sel.drop(columns=cols_drop, inplace=True)

        ## average across seeds, pivot batch size vs sigma
        runs_sel = (
            runs_sel.groupby(["data.batch_size", "model.loss_fn.sigma"])
            .agg({k: ["mean", "sem"] for k in metrics_use})
            .reset_index()
        )

        ## Plot the average of each index as a 2D heatmap of batch size vs sigma

        import matplotlib.pyplot as plt
        import seaborn as sns

        fig, ax = plt.subplots(
            n_rows,
            n_cols,
            figsize=(5 * n_cols, 4 * n_rows),
            gridspec_kw={"hspace": 0.4, "wspace": 0.4},
        )
        ax = ax.flatten()  # Flatten the axes array for easy indexing

        # for i_plot, name_index in enumerate(
        #     metrics_use + ["average"]
        # ):
        #     if name_index == "average":
        #         runs_sel["average"] = runs_sel[metrics_use].mean(axis=1)
        #         df_plot = runs_sel.pivot(
        #             index="data.batch_size", columns="model.loss_fn.sigma", values=name_index
        #         )
        #     else:
        #         df_plot = runs_sel.pivot(
        #             index="data.batch_size", columns="model.loss_fn.sigma", values=(name_index, "mean")
        #         )
        for i_plot, name_index in enumerate(metrics_use):
            df_plot = runs_sel.pivot(
                index="data.batch_size", columns="model.loss_fn.sigma", values=(name_index, "mean")
            )

            sns.heatmap(df_plot, annot=True, fmt=".2f", ax=ax[i_plot], vmin=0, vmax=0.9)
            # sns.heatmap(df_plot, annot=True, fmt=".2f", ax=ax[i_plot], vmin=max(0, min(df_plot.min().min(), df_plot.max().max() - 0.1)), vmax=max(df_plot.max().max(), 0.1))
            ax[i_plot].set_title(name_index, fontsize=10)
            ax[i_plot].set_xlabel("Sigma")
            ax[i_plot].set_ylabel("Batch Size")

        fig.suptitle(
            f"Average of each index across seeds (use_unlabelled_data={use_unlabelled_data}, version={version})",
            fontsize=10,
            y=1.05,
        )

In [ ]:
## Selection

version = "v10"
save_fig = False


criterion_metric = "test_avr_top-dyn_k_index_in_dist_valid"
metrics_table = [
    "test_avr_top-dyn_k_index_aggr_bioclim",
    "test_avr_top-dyn_k_index_aggr_corine_all",
    "test_avr_top-dyn_k_index_aggr_hf",
    "test_avr_top-dyn_k_index_aggr_ukceh",
    "test_avr_top-dyn_k_index_aggr_pollution",
    "test_avr_top-dyn_k_index_aggr_topo",
]
metrics_use = metrics_table + [criterion_metric]
metrics_plot = metrics_table[:-1] + [criterion_metric]  # leave out topo
model_name_map = {
    "AEF labelled": "AEF (labelled)",
    "AEF unlabelled": "AEF (unlabelled)",
    "ClipImgEncoder": "CLIP",
    "DOFACLIP_model": "DOFA-CLIP",
    "GeoClipCoordinateEncoder": "GeoCLIP",
    "GeoRSCLIP_model": "GeoRSCLIP",
    "RemoteCLIP_model": "RemoteCLIP",
    "SkyCLIP_model": "SkyCLIP",
}

columns_name_map = {
    "name_plot": "Model",
    "test_avr_top-dyn_k_index_aggr_bioclim": "Bioclimatic",
    "test_avr_top-dyn_k_index_aggr_corine_all": "Corine LC",
    "test_avr_top-dyn_k_index_aggr_hf": "Human footprint",
    "test_avr_top-dyn_k_index_aggr_ukceh": "UKCEH LC",
    "test_avr_top-dyn_k_index_aggr_pollution": "Pollution",
    "test_avr_top-dyn_k_index_aggr_topo": "Topographic",
    "average": "Group average",
    "test_avr_top-dyn_k_index_in_dist_valid": "Average (subset)",
    "data.dataset.modalities.s2.preprocessing": "S2 preprocessing",
    "model.model_name": "Model Name",
    "data.dataset.modalities.s2.channels": "S2 Channels",
}

n_cols = 3

n_metrics = len(metrics_plot)  # +1 for average
n_rows = n_metrics // n_cols + (n_metrics % n_cols > 0)

for use_unlabelled_data in [True, False]:
    runs_sel = df_dict["aef"].copy()
    if "data.caption_builder.concepts_fname" in runs_sel.columns:
        runs_sel = runs_sel[runs_sel["data.caption_builder.concepts_fname"] == f"{version}.json"]

    if "optimizer.lr" in runs_sel.columns:
        runs_sel = runs_sel[runs_sel["optimizer.lr"] == 0.001]
    if "data.dataset.use_unlabelled_data" in runs_sel.columns:
        runs_sel = runs_sel[runs_sel["data.dataset.use_unlabelled_data"] == use_unlabelled_data]
    else:
        print("No use_unlabelled_data column in runs_sel, skipping filtering by it.")
        if not use_unlabelled_data:
            break
    runs_sel["average"] = runs_sel[metrics_table].mean(axis=1)
    tmp_copy = runs_sel.copy()

    ## average across seeds, pivot batch size vs sigma
    runs_sel = (
        runs_sel.groupby(["data.batch_size", "model.loss_fn.sigma"])
        .agg({k: ["mean", "sem"] for k in metrics_use})
        .reset_index()
    )

    if use_unlabelled_data:
        ind_best_aef_run = runs_sel[(criterion_metric, "mean")].idxmax()
        best_aef_run = runs_sel.loc[ind_best_aef_run]
        ## get best run config from tmp_copy
        best_aef_run_config = tmp_copy[
            np.logical_and(
                tmp_copy["data.batch_size"].values == best_aef_run["data.batch_size"].values,
                (
                    tmp_copy["model.loss_fn.sigma"].values
                    == best_aef_run["model.loss_fn.sigma"].values
                ),
            )
        ]

    fig, ax = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4 * n_cols, 4 * n_rows),
        gridspec_kw={"hspace": 0.4, "wspace": 0.2},
    )
    ax = ax.flatten()  # Flatten the axes array for easy indexing
    ## create horizontal cbar axis under fig
    cbar_ax = fig.add_axes([0.92, 0.15, 0.01, 0.7])  # [left, bottom, width, height]

    for i_plot, name_index in enumerate(metrics_plot):
        df_plot = runs_sel.pivot(
            index="data.batch_size", columns="model.loss_fn.sigma", values=(name_index, "mean")
        )

        sns.heatmap(
            df_plot,
            annot=True,
            fmt=".2f",
            ax=ax[i_plot],
            vmin=0,
            vmax=0.5,
            cbar=i_plot == len(metrics_plot) - 1,
            cbar_ax=cbar_ax if i_plot == len(metrics_plot) - 1 else None,
            cbar_kws=(
                {"label": "Mean value " + r"$I_{\text{top-}k_l}$", "orientation": "vertical"}
                if i_plot == n_metrics - 1
                else None
            ),
        )
        ax[i_plot].set_title(columns_name_map.get(name_index, name_index), fontsize=10)
        if i_plot % n_cols == 0:
            ax[i_plot].set_ylabel("Batch Size")
        else:
            ax[i_plot].set_ylabel("")
        if i_plot // n_cols == n_rows - 1:
            ax[i_plot].set_xlabel("Sigma")
        else:
            ax[i_plot].set_xlabel("")

    while i_plot < len(ax) - 1:
        i_plot += 1
        ax[i_plot].axis("off")

    fig.suptitle(
        f"{'Unlabelled' if use_unlabelled_data else 'Labelled'} AlphaEarth model, with {'mixed' if project == 's2bms_alignment_lv5' else 'separate'} captions, averaged across 3 seeds.",
        fontsize=10,
        y=0.95,
    )

    if save_fig:
        path_save = f"/Users/tplas/repos/ms_aether_biodiv/figs/align_aef_ablation_{'unlabelled' if use_unlabelled_data else 'labelled'}{'_lv5' if project == 's2bms_alignment_lv5' else ''}.pdf"
        plt.savefig(path_save, bbox_inches="tight")

In [ ]:
best_aef_run_config.to_csv(
    os.path.join(
        os.environ["PROJECT_ROOT"], "inference/s2bms/v1/", f"best_aef_run_config_{version}.csv"
    ),
    index=False,
)

# BASELINE

In [ ]:
runs_df = df_dict["baseline"].copy()

cols_params = [
    c
    for c in runs_df.columns
    if "index" not in c
    and c
    not in ["experiment", "run_id", "model/params/total", "model/params/trainable", "group_av"]
]
version = "v10"
save_tables = False
criterion_metric = "test_avr_top-dyn_k_index_in_dist_valid"
metrics_table = [
    "test_avr_top-dyn_k_index_aggr_bioclim",
    "test_avr_top-dyn_k_index_aggr_corine_all",
    "test_avr_top-dyn_k_index_aggr_hf",
    "test_avr_top-dyn_k_index_aggr_ukceh",
    "test_avr_top-dyn_k_index_aggr_pollution",
    "test_avr_top-dyn_k_index_aggr_topo",
]
metrics_use = metrics_table + [criterion_metric]
columns_name_map = {
    "name_plot": "Model",
    "test_avr_top-dyn_k_index_aggr_bioclim": "Biocl.",
    "test_avr_top-dyn_k_index_aggr_corine_all": "Corine",
    "test_avr_top-dyn_k_index_aggr_hf": "HF",
    "test_avr_top-dyn_k_index_aggr_ukceh": "UKCEH",
    "test_avr_top-dyn_k_index_aggr_pollution": "PM2.5",
    "test_avr_top-dyn_k_index_aggr_topo": "Topo.",
    "average": "Av.",
    "test_avr_top-dyn_k_index_in_dist_valid": "Av. (sub)",
    "data.dataset.modalities.s2.preprocessing": "S2 preprocess.",
    "model.model_name": "Model name",
    "data.dataset.modalities.s2.channels": "Chan.",
}


cols_main = ["model._target_", "model.geo_encoder._target_"]
unique_combinations = runs_df[cols_main].drop_duplicates()

runs_df["name_combi"] = runs_df["model._target_"].apply(
    lambda x: x.split(".")[-1].lstrip("build_")
)
runs_df["name_combi"][runs_df["name_combi"] == "TextAlignmentModel"] = runs_df[
    runs_df["name_combi"] == "TextAlignmentModel"
]["model.geo_encoder._target_"].apply(lambda x: x.split(".")[-1])
runs_df = runs_df[runs_df["data.caption_builder.concepts_fname"] == f"{version}.json"]
runs_df["average"] = runs_df[metrics_table].mean(axis=1)
runs_df["model.model_name"] = runs_df["model.model_name"].apply(lambda x: str(x).replace("_", " "))

dict_baseline_tables = {}

for n in runs_df["name_combi"].unique():
    print(
        n,
        len(runs_df[runs_df["name_combi"] == n]),
        "\n",
        runs_df[runs_df["name_combi"] == n]["experiment"].unique(),
        "\n\n\n",
    )
    if n == "GeoClipCoordinateEncoder":
        continue

    runs_df_sel = runs_df[runs_df["name_combi"] == n]
    if n == "ClipImgEncoder":
        runs_df_sel["model.model_name"] = runs_df_sel["experiment"].apply(
            lambda x: x.split("_")[2]
        )
    elif n == "SkyCLIP_model":
        runs_df_sel["model.model_name"] = runs_df_sel["model.model_name"].apply(
            lambda x: x.lstrip("SkyCLIP_")
        )
    runs_df_sel = runs_df_sel[cols_params + metrics_use]
    drop_cols = [c for c in runs_df_sel.columns if runs_df_sel[c].nunique() <= 1]
    runs_df_sel.drop(columns=drop_cols, inplace=True)
    runs_df_sel.rename(columns=columns_name_map, inplace=True)
    runs_df_sel.rename(columns={c: c.replace("_", " ") for c in runs_df_sel.columns}, inplace=True)

    dict_baseline_tables[n] = runs_df_sel
    m = model_name_map[n] if n in model_name_map else n
    if save_tables:
        for c in runs_df_sel.columns:
            if runs_df_sel[c].dtype == "float64":
                # best_index = runs_df_sel[c].idxmax()
                runs_df_sel[c] = runs_df_sel[c].apply(
                    lambda x: f"\\textbf{{{x:.2f}}}" if x == runs_df_sel[c].max() else f"{x:.2f}"
                )
            else:
                runs_df_sel[c] = runs_df_sel[c].apply(lambda x: str(x.replace("_", " ")))
        path_save = (
            f"/Users/tplas/repos/ms_aether_biodiv/tables/align_s2bms_{m.lower()}_ablations.tex"
        )
        runs_df_sel.to_latex(
            path_save,
            index=False,
            escape=False,
            caption=f"Alignment model {m} ablation study.",
            label=f"tab:align_s2bms_{m.lower()}_ablations",
        )

In [ ]:
save_table = True
model_name_map = {
    "AEF_ablations": {
        "AEF labelled, AEF proj.": "AEF, labelled, AEF proj., sep.",
        "AEF unlabelled, AEF proj.": "AEF, unlabelled, AEF proj., sep.",
        "AEF unlabelled, text proj.": "AEF, unlabelled, text proj., sep.",
        "AEF unlabelled, both proj.": "AEF, unlabelled, both proj., sep.",
        "AEF unlabelled, AEF proj., mixed": "AEF, unlabelled, AEF proj., mix.",
        "AEF labelled, AEF proj., mixed": "AEF, labelled, AEF proj., mix.",
    },
    "main": {
        "AEF unlabelled, AEF proj.": "AEF, separate captions",
        "AEF unlabelled, AEF proj., mixed": "AEF, mixed captions",
        "ClipImgEncoder": "CLIP",
        "DOFACLIP_model": "DOFA-CLIP",
        "GeoClipCoordinateEncoder": "GeoCLIP",
        "GeoRSCLIP_model": "GeoRSCLIP",
        "RemoteCLIP_model": "RemoteCLIP",
        "SkyCLIP_model": "SkyCLIP",
    },
}
set_cols = set()
for k, v in dict_baseline_tables.items():
    cols_params = [c for c in v.columns if v[c].dtype != "float64"]
    set_cols.update(cols_params)
    print(f"{k}: {cols_params}")

overview_ablations = {x: [] for x in ["baseline"] + list(set_cols)}
for k, v in dict_baseline_tables.items():
    for c in overview_ablations.keys():
        if c in v.columns:
            overview_ablations[c].append(
                ", ".join([str(x).replace("_", "\_") for x in v[c].unique()])
            )
        elif c == "baseline":
            overview_ablations[c].append(model_name_map["main"][k])
        else:
            overview_ablations[c].append("N/A")

overview_ablations_df = pd.DataFrame(overview_ablations)
overview_ablations_df = overview_ablations_df.rename(
    columns={
        "baseline": "Model",
        "Model name": "Backbone",
        "S2 preprocess.": "Sentinel-2 preprocessing",
        "Chan.": "Channels",
    }
)


if save_table:
    path_save = (
        f"/Users/tplas/repos/ms_aether_biodiv/tables/align_s2bms_overview_baseline_ablations.tex"
    )
    overview_ablations_df.to_latex(
        path_save,
        index=False,
        escape=False,
        caption="\\textbf{Overview of ablation study combinations for the alignment baseline models.} Experiments were done on the S2BMS task, for every combination of (backbone, channels, preprocessing) parameter shown below, per model. SkyCLIP backbones refer to the notation of the original implementation \\citep{wang2024skyscript}. For each model, the best configuration based on the validation metric was selected for Table \\ref{tab:align_s2bms_summary_main}.",
        label=f"tab:align_s2bms_overview_baseline_ablations",
        column_format="lp{5cm}lp{4cm}",
    )

In [ ]:
overview_ablations_df

model_name_map

In [ ]:
print(dict_baseline_tables.keys())
dict_baseline_tables["SkyCLIP_model"]

In [ ]:
## compare to best AEF model

df_sel = df_dict["aef"][df_dict["aef"]["data.caption_builder.concepts_fname"] == f"{version}.json"]
df_sel

In [ ]:
df_dict["baseline"]

## Results combined:

In [ ]:
version = "v10"
save_table = True
criterion_metric = "test_avr_top-dyn_k_index_in_dist_valid"
metrics_table = [
    "test_avr_top-dyn_k_index_aggr_bioclim",
    "test_avr_top-dyn_k_index_aggr_corine_all",
    "test_avr_top-dyn_k_index_aggr_hf",
    "test_avr_top-dyn_k_index_aggr_ukceh",
    "test_avr_top-dyn_k_index_aggr_pollution",
    "test_avr_top-dyn_k_index_aggr_topo",
]
metrics_use = metrics_table + [criterion_metric]

df_best_runs = None
columns_name_map = {
    "name_plot": "Model",
    "test_avr_top-dyn_k_index_aggr_bioclim": "Biocl.",
    "test_avr_top-dyn_k_index_aggr_corine_all": "Corine",
    "test_avr_top-dyn_k_index_aggr_hf": "HF",
    "test_avr_top-dyn_k_index_aggr_ukceh": "UKCEH",
    "test_avr_top-dyn_k_index_aggr_pollution": "PM2.5",
    "test_avr_top-dyn_k_index_aggr_topo": "Topo.",
    "average": "Av.",
    "test_avr_top-dyn_k_index_in_dist_valid": "Av. (sub)",
}
model_name_map = {
    "AEF_ablations": {
        "AEF labelled, AEF proj.": "AEF, labelled, AEF proj., sep.",
        "AEF unlabelled, AEF proj.": "AEF, unlabelled, AEF proj., sep.",
        "AEF unlabelled, text proj.": "AEF, unlabelled, text proj., sep.",
        "AEF unlabelled, both proj.": "AEF, unlabelled, both proj., sep.",
        "AEF unlabelled, AEF proj., mixed": "AEF, unlabelled, AEF proj., mix.",
        "AEF labelled, AEF proj., mixed": "AEF, labelled, AEF proj., mix.",
    },
    "main": {
        "AEF unlabelled, AEF proj.": "AEF, separate captions",
        "AEF unlabelled, AEF proj., mixed": "AEF, mixed captions",
        "ClipImgEncoder": "CLIP",
        "DOFACLIP_model": "DOFA-CLIP",
        "GeoClipCoordinateEncoder": "GeoCLIP",
        "GeoRSCLIP_model": "GeoRSCLIP",
        "RemoteCLIP_model": "RemoteCLIP",
        "SkyCLIP_model": "SkyCLIP",
    },
}


# get best aef labelled and unlabelled.
for use_unlabelled_data in [True, False]:
    runs_sel = df_dict["aef"].copy()
    if "data.caption_builder.concepts_fname" in runs_sel.columns:
        runs_sel = runs_sel[runs_sel["data.caption_builder.concepts_fname"] == f"{version}.json"]
    if "optimizer.lr" in runs_sel.columns:
        runs_sel = runs_sel[runs_sel["optimizer.lr"] == 0.001]
    if "data.dataset.use_unlabelled_data" in runs_sel.columns:
        runs_sel = runs_sel[runs_sel["data.dataset.use_unlabelled_data"] == use_unlabelled_data]
    else:
        if not use_unlabelled_data:
            break
    runs_sel["average"] = runs_sel[metrics_table].mean(axis=1)
    cols_params = ["data.batch_size", "model.loss_fn.sigma"]

    ## average across seeds, pivot batch size vs sigma
    runs_sel = (
        runs_sel.groupby(cols_params).agg({k: ["mean", "sem"] for k in metrics_use}).reset_index()
    )
    runs_sel["name_plot"] = None

    ind_best = runs_sel[(criterion_metric, "mean")].idxmax()
    best_run = runs_sel.loc[ind_best]
    best_run["name_plot"] = f"AEF {'unlabelled' if use_unlabelled_data else 'labelled'}, AEF proj."
    print(
        f"Best AEF {'unlabelled' if use_unlabelled_data else 'labelled'} run: {[(c, best_run[c]) for c in drop_labels]}, {best_run[criterion_metric, 'mean']:.2f}"
    )
    # best_run is a Series here (single row), so drop by index labels (not columns)
    drop_labels = cols_params + [(c, "") for c in cols_params]
    best_run = best_run.drop(
        labels=[c for c in drop_labels if c in best_run.index], errors="ignore"
    )
    if df_best_runs is None:
        df_best_runs = pd.DataFrame([best_run])
    else:
        df_best_runs = pd.concat([df_best_runs, pd.DataFrame([best_run])], ignore_index=True)

# assert False
df_best_runs = pd.concat([df_best_runs, df_best_runs_both], ignore_index=True)
df_best_runs = pd.concat([df_best_runs, df_best_runs_mixed], ignore_index=True)
df_best_runs = pd.concat([df_best_runs, df_best_runs_textual], ignore_index=True)


## collapse mean, sem into a str of (mean ± sem) for each metric
for metric in metrics_use:
    mean_col = (metric, "mean")
    sem_col = (metric, "sem")
    tmp = [
        format_mean_sem(m, s, decimals=2, scale=1.0, math_mode=True)
        for m, s in zip(df_best_runs[mean_col], df_best_runs[sem_col])
    ]
    df_best_runs.drop(columns=[mean_col, sem_col], inplace=True)
    if metric == "test_avr_top-dyn_k_index_aggr_topo":
        continue
    df_best_runs[metric] = tmp

## reset columns to be single-level
df_best_runs.columns = [c[0] if isinstance(c, tuple) else c for c in df_best_runs.columns]

## get best baseline model for each combination of model and geo_encoder, and compare to best AEF model
runs_df = df_dict["baseline"].copy()

cols_params = [
    c
    for c in runs_df.columns
    if "index" not in c
    and c not in ["experiment", "run_id", "model/params/total", "model/params/trainable"]
]
cols_main = ["model._target_", "model.geo_encoder._target_"]
unique_combinations = runs_df[cols_main].drop_duplicates()

runs_df["name_plot"] = runs_df["model._target_"].apply(lambda x: x.split(".")[-1].lstrip("build_"))
runs_df["name_plot"][runs_df["name_plot"] == "TextAlignmentModel"] = runs_df[
    runs_df["name_plot"] == "TextAlignmentModel"
]["model.geo_encoder._target_"].apply(lambda x: x.split(".")[-1])
runs_df = runs_df[runs_df["data.caption_builder.concepts_fname"] == f"{version}.json"]

for name_combi in runs_df["name_plot"].unique():
    runs_sel = runs_df[runs_df["name_plot"] == name_combi].copy()
    runs_sel["average"] = runs_sel[metrics_table].mean(axis=1)
    best_run = runs_sel.loc[runs_sel[(criterion_metric)].idxmax()]
    best_run = best_run[
        df_best_runs.columns
    ]  # Ensure the best_run has the same columns as df_best_runs
    assert all(
        best_run.index == df_best_runs.columns
    ), f"Column mismatch: {best_run.index} vs {df_best_runs.columns}"
    df_best_runs = pd.concat([df_best_runs, pd.DataFrame([best_run])], ignore_index=True)


## Select different model name maps for different tables
for name_map, model_map in model_name_map.items():
    df_best_runs_sel = df_best_runs.copy()
    df_best_runs_sel = df_best_runs_sel[
        df_best_runs_sel["name_plot"].isin(model_map.keys())
    ].copy()
    df_best_runs_sel["name_plot"] = df_best_runs_sel["name_plot"].map(model_map)
    df_best_runs_sel.rename(columns=columns_name_map, inplace=True)

    for c in df_best_runs_sel.columns:
        if c == "Model":
            continue
        df_best_runs_sel[c] = df_best_runs_sel[c].apply(
            lambda x: f"{x:.2f}" if isinstance(x, float) else x
        )

    def get_float_from_str(df, col_ignore=["Model"]):
        df_float = df.copy()
        for c in df_float.columns:
            if c in col_ignore:
                continue
            df_float[c] = df_float[c].apply(
                lambda x: float(x.split(" ")[0]) if isinstance(x, str) and " " in x else float(x)
            )
        return df_float

    df_best_runs_float = get_float_from_str(df_best_runs_sel, col_ignore=["Model"])
    inds_sorted = df_best_runs_float["Av. (sub)"].sort_values(ascending=True).index
    df_best_runs_sel = df_best_runs_sel.loc[inds_sorted].reset_index(drop=True)
    df_best_runs_float = df_best_runs_float.loc[inds_sorted].reset_index(drop=True)

    ## best model per column:
    for c in df_best_runs_float.columns:
        if c == "Model":
            continue
        best_val = df_best_runs_float[c].max()
        inds_best = df_best_runs_float[df_best_runs_float[c] == best_val].index
        df_best_runs_sel.loc[inds_best, c] = df_best_runs_sel.loc[inds_best, c].apply(
            lambda x: f"\\textbf{{{x}}}" if isinstance(x, str) else f"\\textbf{{{x:.2f}}}"
        )

    if name_map == "main":
        caption = "\\textbf{Retrieval scores of the AlphaEarth (AEF) alignment models on the S2BMS task.} For each model, the best configuration after ablation experiments is shown. Scores show the average retrieval $I_{\\text{top-}k_l}$ score for the bioclimatic, CORINE land cover, human footprint (HF), UKCEH land cover, PM2.5 pollution groups, as well as the average across a subset (Av.\ (sub)) of bioclimatic, CORINE and HF columns (used as validation criterion)."
    elif name_map == "AEF_ablations":
        caption = "\\textbf{Ablation study of the AlphaEarth (AEF) alignment model on S2BMS.} The projection network (AEF-only, text-only, both), data (labelled, unlabelled), and caption strategy (separate, mixed) were varied."

    if save_table:
        df_best_runs_sel.to_latex(
            f"/Users/tplas/repos/ms_aether_biodiv/tables/alignment_s2bm_summary_{name_map}.tex",
            index=False,
            escape=False,
            label=f"tab:align_s2bms_summary_{name_map}",
            caption=caption,
        )

In [ ]:
# df_best_runs_textual = df_best_runs.copy()
# df_best_runs_both = df_best_runs.copy()
# df_best_runs_mixed = df_best_runs.copy()

In [ ]:
df_best_runs_sel

## Retrieval vs probe R2

In [ ]:
assert (
    len(best_aef_run_config) == 3 and "k5lcdhoj" in best_aef_run_config["run_id"].values
), f"Unexpected best_aef_run_config"

save_dir = f"/Users/tplas/repos/aether/outputs/alignment/"
DATASET = "s2bms"
unlabelled = True
if DATASET == "satbird-USA-summer":
    filename = f"satbird_ridge_reg_avr_aef_128_to_aux"
else:
    filename = f"s2bms_ridge_reg_avr_aef_256_to_aux_{'unlabelled' if unlabelled else 'labelled'}"

df_r2 = pd.read_csv(os.path.join(save_dir, f"{filename}.csv"))
aux_vars = df_r2["AUX_Variable"].values

list_retrieval_scores = {x: [] for x in ["mean", "sem"]}
list_id_valid = []

for v in aux_vars:
    tmp_valid = None
    if "test-aux_" in v:
        tmp_valid = False
    name_config = f"test_dyn_k_index_{v.lstrip('aux_').lstrip('test-aux_')}"
    found = False
    for suffix in ["_min", "_max"]:
        name_config_suffix = f"{name_config}{suffix}"
        if name_config_suffix in best_aef_run_config.columns:
            name_config = name_config_suffix
            found = True
            if tmp_valid is None:
                tmp_valid = df_r2[df_r2["AUX_Variable"] == v]["Test_R2"].values[0] > 0.5
            break

    if not found:
        list_id_valid.append(False)
        list_retrieval_scores["mean"].append(np.nan)
        list_retrieval_scores["sem"].append(np.nan)
        continue
        #     print(f"Warning: Missing column for aux var {v}, {name_config} in df_best_runs")

    list_id_valid.append(tmp_valid)
    list_retrieval_scores["mean"].append(best_aef_run_config[name_config].values.mean())
    list_retrieval_scores["sem"].append(
        best_aef_run_config[name_config].values.std()
        / np.sqrt(len(best_aef_run_config[name_config].values))
    )

df_r2["retrieval_mean"] = list_retrieval_scores["mean"]
df_r2["retrieval_sem"] = list_retrieval_scores["sem"]
df_r2["is_valid"] = list_id_valid
df_r2 = df_r2[~df_r2["retrieval_mean"].isna()]

In [ ]:
dict_cols

df_r2["var_group"] = df_r2["AUX_Variable"].apply(
    lambda x: next(
        (
            k
            for k, v in dict_cols.items()
            if any([x.lstrip("aux_").lstrip("test-aux_") in s for s in v])
        ),
        "other",
    )
)
df_r2

In [ ]:
df_r2_plot["var_group"].unique()

In [ ]:
save_fig = True
import scipy.stats

df_r2_plot = df_r2[df_r2["var_group"] != "topo"].copy()
fig, ax = plt.subplots(1, 5, figsize=(10, 1.7), gridspec_kw={"hspace": 0.4, "wspace": 0.2})

tasks = {
    "bioclim": "Bioclimatic",
    "corine_all": "CORINE LC",
    "hf": "Human footprint",
    "ukceh": "UKCEH LC",
    "pollution": "Pollution",
}

for iplot, group in enumerate(tasks):
    df_r2_group = df_r2_plot[df_r2_plot["var_group"] == group]
    curr_ax = ax[iplot]
    sns.scatterplot(
        data=df_r2_group,
        x="Test_R2",
        y="retrieval_mean",
        ax=curr_ax,
        color="#B17171" if iplot < 3 else "#81A0D2",
        s=25,
        edgecolor="k",
        alpha=1,
    )

    curr_ax.set_xlabel(r"Test $R^2$")
    curr_ax.set_xlim(min(df_r2_plot["Test_R2"]) - 0.05, 1.05)
    curr_ax.set_ylim(min(df_r2_plot["retrieval_mean"]) - 0.05, 1.05)
    if iplot == 0:
        curr_ax.set_ylabel(r"Av. retrieval $I_{\text{top-}k_l}$")
        curr_ax.set_yticklabels([f"{x:.2f}" for x in curr_ax.get_yticks()])
    else:
        curr_ax.set_ylabel("")
        curr_ax.set_yticklabels([])
    curr_ax.set_xlabel(r"$R^2$ AEF encoding")
    # ax.legend(title='In av. subset', loc='upper left')
    if len(df_r2_group) > 1:
        r, p = scipy.stats.pearsonr(df_r2_group["Test_R2"], df_r2_group["retrieval_mean"])
        r = f", r={r:.2f}, p={p:.2g}"
    else:
        r, p = "", ""
    print(f"Group {group}: r={r}, p={p}")
    curr_ax.set_title(f"{tasks[group]}", fontsize=10, y=1.04)
    for sp in ["top", "right"]:
        curr_ax.spines[sp].set_visible(False)
if save_fig:
    path_save = os.path.join(
        "/Users/tplas/repos/ms_aether_biodiv/figs/", f"{filename}_retrieval_vs_testR2.pdf"
    )
    plt.savefig(path_save, bbox_inches="tight")

In [ ]:
save_fig = False

fig, ax = plt.subplots(figsize=(4, 4))

sns.scatterplot(data=df_r2, x="Test_R2", y="retrieval_mean", hue="var_group", ax=ax)

# for _, row in df_r2.iterrows():
#     ax.text(
#         row["Test_R2"],
#         row["retrieval_mean"],
#         str(row["AUX_Variable"]),
#         fontsize=8,
#         ha="left",
#         va="bottom",
#     )

ax.set_xlabel(r"Test $R^2$")
ax.set_ylabel(r"Retrieval mean $I_{\text{top-}k_l}$")
ax.set_title(f"Correlation between retrieval and test R2 for {DATASET} dataset", fontsize=10)
import scipy.stats

ax.legend(title="In av. subset", loc="upper left")
scipy.stats.pearsonr(df_r2["Test_R2"], df_r2["retrieval_mean"])

if save_fig:
    path_save = os.path.join(
        "/Users/tplas/repos/ms_aether_biodiv/figs/", f"{filename}_retrieval_vs_testR2.pdf"
    )
    plt.savefig(path_save, bbox_inches="tight")

# Plotting

In [ ]:
for col in runs_df.columns:
    print(col)

In [ ]:
cols_of_interest = {
    "experiment": "Modality (best config.)",
    "test_avr_top-dyn_k_index_in_dist_valid": "In distribution (present)",
    "test_avr_top-dyn_k_index_in_dist_unvalid": "In distribution (absent)",
    "test_avr_top-dyn_k_index_ood": "Out-of-distribution",
}

In [ ]:
sub_runs_df = runs_df[cols_of_interest.keys()]

grouped = sub_runs_df.groupby("experiment")

# mean and SEM in one go
summary_mean = grouped.mean(numeric_only=True)
summary_sem = grouped.sem(numeric_only=True)  # pandas has this built in: std/sqrt(n)

# also useful to know how many runs went into each SEM
summary_n = grouped.size().rename("n_runs")

# rename sem columns so they don't clash with mean columns
summary_sem = summary_sem.rename(columns={c: f"{c}_sem" for c in summary_sem.columns})

summary = summary_mean.join(summary_sem).join(summary_n)
summary = summary.sort_values("test_avr_top-dyn_k_index_in_dist_valid", ascending=False)
summary.reset_index(inplace=True)

In [ ]:
summary